# Ribeiro Regeneration Transcriptomic Analysis & Export Pipeline
**Project:** Ribeiro et al. 2019 Replication & Sub-Categorical Filtering Workspace
**Authors:** Alina Krawczynski, Rannyele Passos Ribeiro (powered by Claude)
**Lab:** Özpolat Lab, Washington University in St. Louis, 1 Brookings Drive, St. Louis, MO 63130
**Year:** 2026

---

## Dataset Overview & Biological Context
This dashboard explores whole-body regeneration transcriptomics for two syllid annelids studied in
Ribeiro et al. 2019 (*BMC Genomics* 20:855, `doi:10.1186/s12864-019-6223-y`, "Comparative
transcriptomics in Syllidae (Annelida) indicates that posterior regeneration and regular growth are
comparable, while anterior regeneration is a distinct process"): **Syllis gracilis** (*Sgracilis*) and
**Sphaerosyllis hystrix** (*Shystrix*), each with its own independent *de novo* Trinity assembly and
its own **TMM-normalized gene expression matrix** across three regeneration conditions: **Normal**
(uninjured control), **Anterior regeneration**, and **Posterior regeneration**.

### Understanding the Expression Values
* **Generation:** Reads for each condition were mapped back to that species' own Trinity assembly and
  gene-level counts were TMM-normalized (`*_condition_gene_matrix.TMM.EXPR.matrix`); each column here is
  already a single per-condition value (no replicate columns to average).
* **Normalization:** TMM (trimmed mean of M-values) rescales each condition's library so a handful of very
  highly-expressed transcripts can't skew the comparison between conditions — in plain terms, it corrects
  for differences in sequencing depth and RNA composition so one condition doesn't just look like it has
  "more of everything" purely because it was sequenced more deeply.
* **Interpretation:** Values represent **relative transcript abundance** for that gene in that regeneration
  condition, within that species' own assembly -- *Sgracilis* and *Shystrix* gene IDs are two entirely
  separate ID spaces and are never comparable to one another directly.

### Annotation
Each species has its own **Trinotate functional annotation report**, providing a BLASTX best-hit
description for the subset of genes with a detectable homolog. Annotation lookups and keyword search are
always scoped to the currently selected species.

### Automated Data Cleaning & QC Rules Applied
1. Gene IDs are standardized to strings; no duplicate gene IDs or missing values were found in either
   species' matrix.
2. **Condition column naming was inconsistent between species** (*Sgracilis*: `PS_02_NORMAL_genes.results`,
   `PS_03_ANT_genes.results`, `PS_04_POST_genes.results`; *Shystrix*: `PS_05_Normal`, `PS_06_ANT`,
   `PS_07_POST`) -- both are automatically mapped onto the same three standardized condition labels
   (`Normal`, `Anterior_regen`, `Posterior_regen`) via case-insensitive substring matching, so the rest of
   the dashboard never needs to know about the raw column names.
3. Each species' Trinotate report lists one row per transcript; rows are collapsed to one row per gene
   (matching the expression matrix's gene-level ID space), preferring an annotated isoform when one exists.

---

## Key Features
* **Species Selector:** every step (gene picker, BLAST search, annotation) is scoped to one species at a
  time via a **Species** dropdown, since the two assemblies are independent.
* **Hybrid Gene Input:** search transcripts dynamically using a live-filtering list (hold `Ctrl`/`Cmd` to
  multi-select), paste a raw block of `TR#|c#_g#` IDs, or search by annotation keyword.
* **Dual-Source Search:** find genes either by ID/filter list or by searching their Trinotate BLASTX
  annotation description for a keyword (e.g. "kinase", "laminin").
* **Flexible Condition Filtering:** isolate a single regeneration condition, or hold `Ctrl`/`Cmd` to select
  any custom combination.
* **On-Demand Exporter:** a dual-button layout separates quick visualization from storage, with a
  customizable save-location (type a path or use the folder-browse button; defaults to
  `exported_results/`).
* **Multi-Gene Heatmap:** selecting 2+ genes automatically adds a genes-by-condition, hierarchically
  clustered heatmap (row-normalized so genes at different expression scales stay visually comparable) with
  a colorblind-safe palette toggle, after the individual bar charts.
* **Local BLAST Sequence Search:** paste a nucleotide sequence to find its matching gene ID via a local
  `blastn` search against whichever species' own assembly is selected, with hit-sequence export.

---

## Dashboard User Guide

### Step 1: Choose a Species and Regeneration Conditions
Pick **Syllis gracilis** or **Sphaerosyllis hystrix** in the **Species** dropdown -- the gene picker below
refreshes to that species' own gene list. Hold `Ctrl` (or `Cmd` on Mac) in the **Conditions** box to select
any custom combination; all three are selected by default.

### Step 2: Choose Target Genes (Use A, B, or C!)
* **Option A (Filter List):** type into **Filter List** to shrink the selection panel in real-time, then
  click (or Ctrl/Cmd-click) gene IDs.
* **Option B (Paste Box):** drop a raw text block of `TR#|c#_g#` IDs copied from a spreadsheet straight into
  the box.
* **Option C (Search by Keyword):** type a keyword to find every gene in the selected species whose
  Trinotate BLASTX annotation description contains it.

### Step 3: Run and/or Save
* Click **🎯 Generate Visuals (View Only)** to render bar charts, the multi-gene heatmap, and annotation
  results instantly on-screen.
* Click **💾 Export Active Results (Save CSV/PDF)** to save high-resolution vector assets and an annotation
  CSV to your chosen folder (defaults to `exported_results/`).

In [ ]:
# ============================================================================================
#  DEPENDENCY CHECK -- run this cell first.
#  Installs any Python packages this notebook needs that aren't already present, then you're
#  done. Safe to re-run (does nothing if everything is already installed). To install by hand
#  instead, run this in a terminal / Anaconda Prompt:
#      pip install pandas numpy matplotlib seaborn scipy ipywidgets
# ============================================================================================
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy', 'ipywidgets']

_missing = [p for p in REQUIRED_PACKAGES if importlib.util.find_spec(p) is None]
if _missing:
    print("Installing missing packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
    print("\nInstall finished. If an import fails in a later cell, restart the kernel")
    print("(menu: Kernel -> Restart Kernel) and run the notebook again from the top.")
else:
    print("All required packages are already installed -- you're good to go.")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Standardized regeneration-condition labels, used for both species regardless of their raw
# column-naming differences.
STANDARD_CONDITIONS = ["Normal", "Anterior_regen", "Posterior_regen"]

def _standardize_columns(df):
    """Maps each species' inconsistently-named sample columns (e.g.
    'PS_02_NORMAL_genes.results' vs 'PS_05_Normal') onto the same standardized condition labels,
    via case-insensitive substring matching rather than hardcoding either naming convention."""
    rename_map = {}
    for col in df.columns:
        col_upper = col.upper()
        if "NORMAL" in col_upper:
            rename_map[col] = "Normal"
        elif "ANT" in col_upper:
            rename_map[col] = "Anterior_regen"
        elif "POST" in col_upper:
            rename_map[col] = "Posterior_regen"
    return df.rename(columns=rename_map)[STANDARD_CONDITIONS]

def _load_species_matrix(tmm_path):
    df = pd.read_csv(tmm_path, sep="\t", index_col=0)
    df.index = df.index.astype(str)
    return _standardize_columns(df)

def _load_annotation(trimmed_path):
    """Loads the pre-processed, gene-level annotation file for one species. This packaged copy
    ships a pre-collapsed, pre-parsed annotation table instead of the raw Trinotate report: the
    original report lists one row per TRANSCRIPT (with 16 columns including RNAMMER, Pfam,
    SignalP, TmHMM, eggnog, Kegg, multiple GO-term columns, and full nucleotide/peptide sequence
    columns per row) at 616MB/438MB for the two species -- far too large to host on GitHub. This
    file instead already has one row per GENE (matching the expression matrix's gene-level ID
    space, preferring an annotated isoform when one exists) with just the parsed BLASTX best-hit
    description, gzip-compressed down to ~1MB per species with identical annotation content to
    what the original notebook produced at runtime."""
    ann = pd.read_csv(trimmed_path, low_memory=False)
    return ann.set_index("gene_id")

# 2. Per-species data + own local BLAST database + own pre-processed Trinotate annotation (each
# species has its own independent Trinity assembly, so gene IDs, BLAST hits, and annotations are
# never comparable across species).
SPECIES_DATA = {
    "Syllis gracilis (Sgracilis)": {
        "df": _load_species_matrix("Sgracilis_condition_gene_matrix.TMM.EXPR.matrix"),
        "annotation": _load_annotation("Sgracilis_annotation_trimmed.csv.gz"),
        "blast_db": os.path.join("blast_db", "ribeiro_sgracilis"),
    },
    "Sphaerosyllis hystrix (Shystrix)": {
        "df": _load_species_matrix("Shystrix_condition_gene_matrix.TMM.EXPR.matrix"),
        "annotation": _load_annotation("Shystrix_annotation_trimmed.csv.gz"),
        "blast_db": os.path.join("blast_db", "ribeiro_shystrix"),
    },
}

# 3. Precompute each species' sorted gene-ID list + a lowercase mirror ONCE here (up to 309,576
# genes for Sgracilis) -- the "Filter List" box used to re-sort + re-lower the full list from
# scratch on every keystroke, which is what made typing in that box feel unresponsive.
for _name, _d in SPECIES_DATA.items():
    _sorted_ids = sorted(_d["df"].index.astype(str))
    _d["gene_ids_sorted"] = _sorted_ids
    _d["gene_ids_lower"] = [g.lower() for g in _sorted_ids]

for _name, _d in SPECIES_DATA.items():
    _n_annotated = int(_d["annotation"]["blastx_description"].notna().sum())
    print(f"✅ {_name}: {len(_d['df'])} genes across {len(STANDARD_CONDITIONS)} regeneration conditions "
          f"({_n_annotated} with a Trinotate BLASTX annotation).")

---

## 🔬 BLAST Sequence Search
Have a nucleotide sequence from elsewhere and want to know if/where it shows up in one of these two
assemblies? Pick a **Species** below, then paste the sequence to BLAST it against that species' own
Trinity transcriptome (a local `blastn` search — no internet required). Any matching `TR#|c#_g#` gene ID
found can then be copied into the **Paste Box** in Step 2 below to view that gene's expression profile.
Click **💾 Export Hit Sequences** to save the hit table (CSV) and the matched sequences plus your original
query (FASTA) to your chosen folder.

**Setup (optional — the rest of the dashboard works without this):** the sequence search needs
the free NCBI BLAST+ toolkit plus a small search database you build once from the reference
transcriptome. Step-by-step instructions — install BLAST+, get the reference FASTA, and build
the database into a `blast_db/` folder next to this notebook — are in the comment at the top of
the next code cell. The notebook finds BLAST+ automatically once it is installed; until then this
section just prints a short reminder and everything else in the notebook is unaffected.

In [ ]:
import shutil
import subprocess
import tempfile
import re

# --- Sequence -> Gene ID search: paste any raw nucleotide sequence and find the best-matching
# transcript(s) in the selected species' own Trinity assembly via a local blastn search.
#
# SETUP (only needed once per machine): install BLAST+ from
# https://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/ , then for EACH species rewrite headers so
# the TR#_c#_g#_i# transcript ID is the first token (required for ID-based lookup/export, since the
# original "TR#|c#_g#_i#" header uses "|", which -parse_seqids reserves for its own structured-ID
# format and would otherwise silently rewrite to "TR#:c#_g#_i#"):
#   python -c "
#   import re
#   with open('Sgracilis_transcriptome.fasta') as fin, open('renamed.fasta', 'w') as fout:
#       for line in fin:
#           if line.startswith('>'):
#               m = re.match(r'>(\S+)', line)
#               fout.write(f'>{m.group(1).replace(\"|\", \"_\", 1)}{line[len(m.group(0)):]}' if m else line)
#           else:
#               fout.write(line)
#   "
# then build the database with -parse_seqids so sequences can be retrieved by that ID:
#   makeblastdb -in renamed.fasta -dbtype nucl -parse_seqids -out blast_db/ribeiro_sgracilis
# (repeat for Shystrix-transcriptome.fasta -> ribeiro_shystrix). The notebook auto-detects BLAST+ on your PATH (or set
# BLAST_BIN_DIR below); the two databases go in a "blast_db" folder next to this notebook
# (SPECIES_DATA above already points there).
# Once built, each database is fully self-contained -- the original transcriptome FASTA is not
# needed for searching OR sequence export, and can be deleted to save space if desired.

# --- BLAST+ location (auto-detected) ---------------------------------------------------------
# By default this notebook finds the BLAST+ programs (blastn, blastdbcmd, makeblastdb) on your
# system PATH -- if you installed BLAST+ the normal way, leave BLAST_BIN_DIR = "" and it just
# works. If BLAST+ lives somewhere not on your PATH, set BLAST_BIN_DIR to the folder that holds
# the blastn / blastdbcmd / makeblastdb executables.
BLAST_BIN_DIR = ""   # e.g. r"C:\Program Files\NCBI\blast-2.17.0+\bin"  or  "/usr/local/ncbi/blast/bin"

# Each species' search-database path is set in SPECIES_DATA (the cell above); by default those
# point at a "blast_db" folder next to this notebook. Build them once from each transcriptome --
# see the SETUP note above. This whole BLAST section is optional; the rest of the dashboard
# works fine without it.

def _find_blast_tool(tool_name):
    """Locate a BLAST+ executable: honour BLAST_BIN_DIR if it is set, otherwise search the
    system PATH. Returns the full path or None. shutil.which adds the right extension (.exe)
    on Windows automatically."""
    if BLAST_BIN_DIR:
        found = shutil.which(tool_name, path=BLAST_BIN_DIR)
        if found:
            return found
    return shutil.which(tool_name)

# Cache of the most recent search, so the export button can act on whatever was last searched
_CURRENT_BLAST_QUERY = None
_CURRENT_BLAST_HITS = None

def _to_fasta(raw_text):
    """Wrap raw pasted text into a single-record FASTA query -- accepts either bare sequence
    (any whitespace/line breaks) or an already-FASTA-formatted block with its own header."""
    raw_text = raw_text.strip()
    if raw_text.startswith(">"):
        return raw_text
    seq = "".join(raw_text.split())
    return f">user_query\n{seq}"

def run_blast_search(raw_sequence, db_path, target_df, max_hits=10):
    """BLASTs a pasted nucleotide sequence against the given species' transcriptome and returns a
    results DataFrame with one row per hit. Gene_ID is derived from the rewritten transcript ID
    (TR#_c#_g#_i#) by dropping the isoform suffix and restoring the "|" separator used in the
    expression matrix (TR#|c#_g#)."""
    blastn_exe = _find_blast_tool("blastn")
    if blastn_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    if not os.path.isfile(db_path + ".nin"):
        print(f"❌ BLAST database not found at {db_path} -- see the setup note above this cell.")
        return None

    fasta_query = _to_fasta(raw_sequence)
    seq_only = "".join(line for line in fasta_query.splitlines() if not line.startswith(">"))
    if len(seq_only) < 15:
        print("❌ Query sequence looks too short (or empty) to BLAST -- paste at least ~15 bases.")
        return None

    with tempfile.NamedTemporaryFile(mode="w", suffix=".fasta", delete=False) as f:
        f.write(fasta_query)
        query_path = f.name

    try:
        result = subprocess.run(
            [blastn_exe, "-query", query_path, "-db", db_path,
             "-outfmt", "6 sseqid pident length evalue bitscore stitle",
             "-max_target_seqs", str(max_hits)],
            capture_output=True, text=True, timeout=60
        )
    finally:
        os.remove(query_path)

    if result.returncode != 0:
        print("❌ blastn reported an error:")
        print(result.stderr.strip())
        return None

    lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
    if not lines:
        print("🛑 No BLAST hits found for this sequence in this species' transcriptome.")
        return None

    rows = []
    for line in lines:
        sseqid, pident, length, evalue, bitscore, stitle = line.split("\t", 5)
        m = re.match(r"(TR\d+)_(c\d+_g\d+)_i\d+", sseqid)
        gene_id = f"{m.group(1)}|{m.group(2)}" if m else None
        rows.append({
            "Gene_ID": gene_id,
            "In_Dashboard": bool(gene_id) and gene_id in target_df.index,
            "Transcript_Hit": sseqid,
            "%Identity": float(pident),
            "Alignment_Length": int(length),
            "E-value": float(evalue),
            "Bit_Score": float(bitscore),
        })

    df_hits = pd.DataFrame(rows).sort_values("Bit_Score", ascending=False).reset_index(drop=True)
    return df_hits

def get_sequence_from_db(seq_id, db_path):
    """Retrieves a sequence's full FASTA record directly from the BLAST database via blastdbcmd
    -- works entirely from the compiled database, without needing the original transcriptome
    FASTA on disk."""
    blastdbcmd_exe = _find_blast_tool("blastdbcmd")
    if blastdbcmd_exe is None:
        print("❌ BLAST+ not found on your system. Install it (see the SETUP note above) or set")
        print("   BLAST_BIN_DIR at the top of this cell. The rest of the dashboard works without BLAST.")
        return None
    result = subprocess.run(
        [blastdbcmd_exe, "-db", db_path, "-entry", seq_id],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

def export_blast_hits(query_sequence, df_hits, db_path, save_dir="exported_results"):
    """Saves the current BLAST hit table (CSV) and the matched sequences, retrieved straight
    from the database, as a multi-FASTA file (also including the original query sequence for
    reference) to save_dir."""
    if df_hits is None or df_hits.empty:
        print("❌ Nothing to export -- run a BLAST search with results first!")
        return

    os.makedirs(save_dir, exist_ok=True)
    df_hits.to_csv(os.path.join(save_dir, "blast_hits.csv"), index=False)
    print(f"💾 [SAVED] BLAST hits table exported to '{save_dir}/blast_hits.csv'")

    fasta_records = [_to_fasta(query_sequence).replace(">user_query", ">query_sequence", 1)]
    for seq_id in df_hits["Transcript_Hit"]:
        record = get_sequence_from_db(seq_id, db_path)
        if record:
            fasta_records.append(record)

    with open(os.path.join(save_dir, "blast_hit_sequences.fasta"), "w") as f:
        f.write("\n".join(fasta_records) + "\n")
    print(f"💾 [SAVED] Query + {len(fasta_records) - 1} hit sequence(s) exported to '{save_dir}/blast_hit_sequences.fasta'")

In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder-picker so the user can point at an existing folder directly
    instead of typing/remembering a path -- falls back to leaving the text box untouched if the
    dialog is cancelled or unavailable. Dispatches to a different native dialog per OS: macOS uses
    AppleScript's "choose folder" (run via osascript, in its own process); Windows uses tkinter;
    Linux tries zenity/kdialog first and falls back to tkinter. This is deliberately NOT just
    tkinter everywhere -- opening a tkinter dialog *inside* a live Jupyter kernel process reliably
    crashes/restarts the kernel on macOS (Tk's window-server integration collides with the
    kernel's own Cocoa/asyncio event loop there). Running the picker as a separate process, as the
    macOS/Linux branches do, sidesteps that entirely; tkinter stays fine in-process on Windows."""
    def _browse(b):
        import platform
        import subprocess
        import shutil as _shutil

        system = platform.system()
        folder = None
        try:
            if system == "Darwin":
                result = subprocess.run(
                    ["osascript", "-e",
                     'POSIX path of (choose folder with prompt "Choose a folder to save results into")'],
                    capture_output=True, text=True, timeout=120
                )
                if result.returncode == 0:
                    folder = result.stdout.strip()
            elif system == "Windows":
                import tkinter as tk
                from tkinter import filedialog
                root = tk.Tk()
                root.withdraw()
                root.attributes("-topmost", True)
                folder = filedialog.askdirectory(title="Choose a folder to save results into")
                root.destroy()
            else:
                for cmd in (["zenity", "--file-selection", "--directory",
                             "--title=Choose a folder to save results into"],
                            ["kdialog", "--getexistingdirectory"]):
                    if _shutil.which(cmd[0]):
                        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
                        if result.returncode == 0:
                            folder = result.stdout.strip()
                        break
                else:
                    import tkinter as tk
                    from tkinter import filedialog
                    root = tk.Tk()
                    root.withdraw()
                    folder = filedialog.askdirectory(title="Choose a folder to save results into")
                    root.destroy()
        except Exception as e:
            print(f"⚠️ Couldn't open a folder picker ({e}) -- type the path into the box above instead.")
            folder = None

        if folder:
            target_box.value = folder
    return _browse

blast_species_dropdown = widgets.Dropdown(options=list(SPECIES_DATA.keys()), description="Species:", layout=widgets.Layout(width="420px"))
blast_query_box = widgets.Textarea(
    placeholder="Paste a raw nucleotide sequence here (with or without a '>' FASTA header)...",
    description="Sequence:",
    layout=widgets.Layout(width="600px", height="120px")
)
blast_button = widgets.Button(description="🔬 Run BLAST Search", button_style="warning", layout=widgets.Layout(width="270px", height="40px"))
blast_export_button = widgets.Button(description="💾 Export Hit Sequences", button_style="info", layout=widgets.Layout(width="270px", height="40px"))
blast_save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
blast_browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
blast_browse_button.on_click(_make_browse_handler(blast_save_location_box))
blast_output = widgets.Output()

def on_blast_clicked(b):
    global _CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS
    with blast_output:
        clear_output(wait=True)
        raw_sequence = blast_query_box.value.strip()
        if not raw_sequence:
            print("❌ Paste a sequence first!")
            return

        species = blast_species_dropdown.value
        db_path = SPECIES_DATA[species]["blast_db"]
        target_df = SPECIES_DATA[species]["df"]

        print(f"🔬 Running BLAST search against {species}'s transcriptome...")
        df_hits = run_blast_search(raw_sequence, db_path, target_df)
        _CURRENT_BLAST_QUERY = raw_sequence
        _CURRENT_BLAST_HITS = df_hits
        if df_hits is None:
            return

        print(f"\n✅ {len(df_hits)} hit(s) found (best match first):")
        display(df_hits)

        best_ids = df_hits.loc[df_hits["In_Dashboard"], "Gene_ID"].dropna().unique().tolist()
        if best_ids:
            print(f"\n📋 Copy into the 'Paste Box' below to view expression profiles: {', '.join(best_ids)}")
        else:
            print("\n⚠️ None of the matched transcripts have a corresponding entry in the expression matrix below.")

def on_blast_export_clicked(b):
    with blast_output:
        clear_output(wait=True)
        species = blast_species_dropdown.value
        db_path = SPECIES_DATA[species]["blast_db"]
        _base_dir = blast_save_location_box.value.strip() or "exported_results"
        save_dir = os.path.join(_base_dir, datetime.now().strftime("search_%Y-%m-%d_%H-%M-%S"))
        print(f"📁 Saving this BLAST search to its own new folder: {save_dir}/")
        export_blast_hits(_CURRENT_BLAST_QUERY, _CURRENT_BLAST_HITS, db_path, save_dir=save_dir)

blast_button.on_click(on_blast_clicked)
blast_export_button.on_click(on_blast_export_clicked)

print("🔬 BLAST Sequence Search Ready:")
display(widgets.VBox([
    widgets.HTML("<i>Pick a species, then paste a sequence of interest to find its matching gene ID in that species' dataset.</i>"),
    blast_species_dropdown,
    blast_query_box,
    widgets.HBox([blast_save_location_box, blast_browse_button]),
    widgets.HBox([blast_button, blast_export_button]),
    blast_output
]))

In [ ]:
import re
import numpy as np
from IPython.display import display, HTML

# Global tracking variables so the export button knows what is currently active
_CURRENT_SPECIES = None
_CURRENT_GENES = []
_CURRENT_CONDITIONS = []

def _safe_filename(gene_id):
    """Strips characters Windows forbids in filenames (e.g. the '|' in TR#|c#_g# gene IDs), so
    no gene ID can ever crash plt.savefig."""
    return re.sub(r'[\\/:*?"<>|]', "_", gene_id)

def _bar_palette(cmap, n):
    """Builds the palette used for the per-condition bars. The default keeps the original 'mako'
    look (independent of the heatmap's red-blue default) since bar color here is purely
    decorative, not tied to a z-score axis. Non-default (colorblind) choices reuse the same
    diverging colormap picked for the heatmap, sampled from its saturated half only -- diverging
    colormaps (PuOr, BrBG) have a washed-out near-white color at their exact center, which would
    otherwise make a bar look broken/invisible if landed on."""
    if cmap == "RdBu_r":
        return "mako"
    cmap_obj = plt.get_cmap(cmap)
    if cmap in ("PuOr", "BrBG"):
        positions = np.linspace(0.60, 0.95, max(n, 1))
    else:
        positions = np.linspace(0.15, 0.85, max(n, 1))
    return [cmap_obj(p) for p in positions]

def _lookup_annotation(gene_id, species_name):
    """Checks the given species' own Trinotate BLASTX annotation for this gene."""
    ann = SPECIES_DATA[species_name]["annotation"]
    if gene_id in ann.index:
        desc = ann.loc[gene_id, "blastx_description"]
        if isinstance(desc, str):
            return [{"Gene_ID": gene_id, "Source": "Trinotate BLASTX", "Description": desc}]
    return []

def _keyword_regex(keyword):
    """Left-word-boundary pattern: the keyword must start at a word boundary (so a short keyword
    can't silently match as a fragment buried inside an unrelated word -- e.g. "vasa" matching
    "extravasation"), but is still free to be a PREFIX of a longer word (so "kinase" still
    matches "kinases", "kinase-like", etc.)."""
    return r"\b" + re.escape(keyword)

def _search_annotations_by_keyword(keyword, species_name):
    """Finds every gene in the given species whose Trinotate BLASTX description contains the
    given keyword as a word-initial match (see _keyword_regex), returning gene IDs actually
    searchable in this dashboard (i.e. present in that species' expression matrix)."""
    keyword = keyword.strip()
    if not keyword:
        return []
    pattern = _keyword_regex(keyword)
    ann = SPECIES_DATA[species_name]["annotation"]
    mask = ann["blastx_description"].astype(str).str.contains(pattern, case=False, na=False, regex=True)
    matched_ids = ann.index[mask].astype(str).tolist()
    df = SPECIES_DATA[species_name]["df"]
    matched_ids = [g for g in matched_ids if g in df.index]
    return sorted(matched_ids)

def plot_multi_genes(species_name, gene_list, conditions_chosen=None):
    """Resolves the gene/condition selection (scoped to one species) and stores it in
    _CURRENT_SPECIES/_CURRENT_GENES/_CURRENT_CONDITIONS. Call _render_bar_charts() afterward to
    actually display output -- split out so the caller controls display order (heatmap, then
    plots)."""
    global _CURRENT_SPECIES, _CURRENT_GENES, _CURRENT_CONDITIONS

    df = SPECIES_DATA[species_name]["df"]
    conditions = [c for c in STANDARD_CONDITIONS if not conditions_chosen or c in conditions_chosen]
    if not conditions:
        conditions = STANDARD_CONDITIONS

    _CURRENT_SPECIES = species_name
    _CURRENT_GENES = [g.strip() for g in gene_list if g.strip() in df.index]
    _CURRENT_CONDITIONS = conditions

def _render_bar_charts(cmap="RdBu_r"):
    df = SPECIES_DATA[_CURRENT_SPECIES]["df"]
    for gene_id in _CURRENT_GENES:
        gene_data = df.loc[gene_id][_CURRENT_CONDITIONS]

        fig, ax = plt.subplots(figsize=(max(3, len(gene_data.index) * 1.3), 4.5))
        sns.barplot(
            x=gene_data.index,
            y=gene_data.values.astype(float),
            hue=gene_data.index,
            palette=_bar_palette(cmap, len(gene_data.index)),
            legend=False,
            width=0.6
        )

        plt.title(f"Expression Profile: {gene_id} ({_CURRENT_SPECIES.split(' (')[0]})", fontsize=16, fontweight="bold", pad=12)
        plt.ylabel("TMM-Normalized Expression", fontsize=14, fontweight="bold")
        plt.xlabel("Regeneration Condition", fontsize=14, fontweight="bold")
        plt.xticks(rotation=20, ha="right", fontsize=13)
        plt.yticks(fontsize=13)
        sns.despine()
        plt.tight_layout()
        plt.show()

def _print_annotations(gene_list):
    print("\n📋 GENE ANNOTATION RESULTS:")
    for gene_id in gene_list:
        gene_id = gene_id.strip()
        rows = _lookup_annotation(gene_id, _CURRENT_SPECIES)
        if rows:
            print(f"\nAnnotations for Search ID: {gene_id}")
            display(pd.DataFrame(rows))
        else:
            print(f"\nAnnotations for Search ID: {gene_id} -> 🛑 [No annotation found]")

# --- SEPARATE EXPORT EXECUTION FUNCTION ---
def export_cached_results(save_dir="exported_results", cmap="RdBu_r"):
    if not _CURRENT_GENES:
        print("❌ Nothing available to save. Please run a successful query search first!")
        return

    df = SPECIES_DATA[_CURRENT_SPECIES]["df"]
    os.makedirs(save_dir, exist_ok=True)

    # 1. Export current annotations to CSV
    all_rows = []
    for gene_id in _CURRENT_GENES:
        all_rows.extend(_lookup_annotation(gene_id, _CURRENT_SPECIES))

    if all_rows:
        pd.DataFrame(all_rows).to_csv(os.path.join(save_dir, "selected_gene_annotations.csv"), index=False)
        print(f"💾 [SAVED] Annotation CSV created inside '{save_dir}/' folder.")

    # 2. Export current graphs to vector PDFs
    for gene_id in _CURRENT_GENES:
        gene_data = df.loc[gene_id][_CURRENT_CONDITIONS]

        fig, ax = plt.subplots(figsize=(max(3, len(gene_data.index) * 1.3), 4.5))
        sns.barplot(x=gene_data.index, y=gene_data.values.astype(float), hue=gene_data.index,
                    palette=_bar_palette(cmap, len(gene_data.index)), legend=False, width=0.6)
        plt.title(f"Expression Profile: {gene_id} ({_CURRENT_SPECIES.split(' (')[0]})", fontsize=16, fontweight="bold", pad=12)
        plt.ylabel("TMM-Normalized Expression", fontsize=14, fontweight="bold")
        plt.xlabel("Regeneration Condition", fontsize=14, fontweight="bold")
        plt.xticks(rotation=20, ha="right", fontsize=13)
        plt.yticks(fontsize=13)
        sns.despine()
        plt.tight_layout()

        plt.savefig(os.path.join(save_dir, f"expression_profile_{_safe_filename(gene_id)}.pdf"), format="pdf", dpi=300, bbox_inches="tight")
        plt.close()

    print(f"💾 [SAVED] Vector PDF plots for {_CURRENT_GENES} generated inside '{save_dir}/' folder.")

In [ ]:
from scipy.cluster.hierarchy import linkage

# --- Multi-gene comparison view: hierarchically-clustered heatmap. Only meaningful with 2+ genes
# selected, so it's skipped automatically for single-gene searches. Both genes (rows) and
# conditions (columns) are hierarchically clustered so genes/conditions with similar profiles end
# up next to each other.

CMAP_OPTIONS = {
    "Default": "RdBu_r",
    "Purple-Orange (colorblind-safe)": "PuOr",
    "Brown-Teal (colorblind-safe)": "BrBG",
    "Cividis (colorblind-safe, single-hue)": "cividis",
}

def _zscore_matrix(gene_ids, cols, matrix):
    sub = matrix.loc[gene_ids, cols].astype(float)
    # Row z-score so genes with very different absolute expression levels are still
    # visually comparable on the same color scale.
    row_std = sub.std(axis=1).replace(0, 1)
    z = sub.sub(sub.mean(axis=1), axis=0).div(row_std, axis=0)
    return sub, z

def _render_heatmap(gene_ids, cols, matrix, value_label="Expression", cmap="RdBu_r"):
    sub, z = _zscore_matrix(gene_ids, cols, matrix)

    # Drop any gene/condition with zero valid data among the current selection -- otherwise a
    # fully-blank row or column (common when only some picked genes have data for some of the
    # picked conditions) pads the heatmap out with wasted, confusing empty space.
    z = z.loc[z.notna().any(axis=1), z.notna().any(axis=0)]
    gene_ids = list(z.index)
    cols = list(z.columns)

    z_for_clustering = z.fillna(0)
    row_linkage = linkage(z_for_clustering, method="average") if len(gene_ids) > 1 else None
    col_linkage = linkage(z_for_clustering.T, method="average") if len(cols) > 1 else None

    g = sns.clustermap(
        z, cmap=cmap, center=0, mask=z.isna(),
        row_linkage=row_linkage, col_linkage=col_linkage,
        row_cluster=len(gene_ids) > 1, col_cluster=len(cols) > 1,
        linewidths=0.5, annot=False,
        figsize=(max(6, len(cols) * 0.9) + 2.2, max(4, len(gene_ids) * 1.3 + 2)),
        cbar_pos=(0.90, 0.86, 0.025, 0.10), dendrogram_ratio=(0.15, 0.15),
    )

    # sns.clustermap lets the heatmap/col-dendrogram extend almost to the figure's right edge
    # regardless of cbar_pos (the colorbar floats independently, on top), so it silently overlaps
    # the heatmap's top-right corner. Directly shrink the heatmap + column-dendrogram axes to end
    # before the colorbar's left edge -- NOT via fig.subplots_adjust(), which was found (by direct
    # testing) to trigger an internal seaborn recompute that snaps cbar_pos back to its own default
    # top-left position instead of respecting the value passed in. The colorbar is also placed
    # above the heatmap (not beside its top rows) so it never collides with row labels either.
    cbar_left, margin = 0.90, 0.02
    for ax in (g.ax_heatmap, g.ax_col_dendrogram):
        pos = ax.get_position()
        ax.set_position([pos.x0, pos.y0, (cbar_left - margin) - pos.x0, pos.height])

    g.ax_heatmap.set_xlabel("Regeneration Condition", fontsize=16, fontweight="bold")
    g.ax_heatmap.set_ylabel("")
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=20, ha="right", fontsize=14)
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=-30, ha="left", va="center", fontsize=13)
    g.ax_cbar.set_title("Row\nz-score", fontsize=11, fontweight="bold")
    g.ax_cbar.tick_params(labelsize=8)
    g.fig.suptitle(f"Expression Heatmap ({value_label}, hierarchically clustered)",
                   fontsize=17, fontweight="bold", x=0.42, ha="center", y=1.08)
    return g.fig

def _has_any_valid_data(gene_ids, cols, matrix):
    cols = list(dict.fromkeys(cols))  # defensive: guard against any duplicate column labels
    _, z = _zscore_matrix(gene_ids, cols, matrix)
    return bool(z.notna().to_numpy().sum())

def render_multi_gene_views(gene_ids, cols, matrix, value_label="TMM-Normalized Expression", cmap="RdBu_r"):
    """Call after plot_multi_genes(): shows the clustered heatmap when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    cols = list(dict.fromkeys(cols))
    if not _has_any_valid_data(gene_ids, cols, matrix):
        print("\n⚠️ No valid data for the selected genes across the chosen conditions -- heatmap skipped.")
        return
    print("\n📊 MULTI-GENE COMPARISON HEATMAP:")
    _render_heatmap(gene_ids, cols, matrix, value_label, cmap)
    plt.show()

def save_multi_gene_views(gene_ids, cols, matrix, value_label="TMM-Normalized Expression", cmap="RdBu_r", save_dir="exported_results"):
    """Call after export_cached_results(): saves the heatmap as a PDF when 2+ genes are selected."""
    if len(gene_ids) < 2:
        return
    cols = list(dict.fromkeys(cols))
    if not _has_any_valid_data(gene_ids, cols, matrix):
        print("\n⚠️ No valid data for the selected genes across the chosen conditions -- heatmap skipped.")
        return
    os.makedirs(save_dir, exist_ok=True)
    fig = _render_heatmap(gene_ids, cols, matrix, value_label, cmap)
    fig.savefig(os.path.join(save_dir, "multi_gene_heatmap.pdf"), format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"💾 [SAVED] Heatmap PDF generated inside '{save_dir}/' folder.")

In [ ]:
import ipywidgets as widgets
from datetime import datetime
from IPython.display import display, clear_output

def _make_browse_handler(target_box):
    """Opens a native OS folder-picker so the user can point at an existing folder directly
    instead of typing/remembering a path -- falls back to leaving the text box untouched if the
    dialog is cancelled or unavailable. Dispatches to a different native dialog per OS: macOS uses
    AppleScript's "choose folder" (run via osascript, in its own process); Windows uses tkinter;
    Linux tries zenity/kdialog first and falls back to tkinter. This is deliberately NOT just
    tkinter everywhere -- opening a tkinter dialog *inside* a live Jupyter kernel process reliably
    crashes/restarts the kernel on macOS (Tk's window-server integration collides with the
    kernel's own Cocoa/asyncio event loop there). Running the picker as a separate process, as the
    macOS/Linux branches do, sidesteps that entirely; tkinter stays fine in-process on Windows."""
    def _browse(b):
        import platform
        import subprocess
        import shutil as _shutil

        system = platform.system()
        folder = None
        try:
            if system == "Darwin":
                result = subprocess.run(
                    ["osascript", "-e",
                     'POSIX path of (choose folder with prompt "Choose a folder to save results into")'],
                    capture_output=True, text=True, timeout=120
                )
                if result.returncode == 0:
                    folder = result.stdout.strip()
            elif system == "Windows":
                import tkinter as tk
                from tkinter import filedialog
                root = tk.Tk()
                root.withdraw()
                root.attributes("-topmost", True)
                folder = filedialog.askdirectory(title="Choose a folder to save results into")
                root.destroy()
            else:
                for cmd in (["zenity", "--file-selection", "--directory",
                             "--title=Choose a folder to save results into"],
                            ["kdialog", "--getexistingdirectory"]):
                    if _shutil.which(cmd[0]):
                        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
                        if result.returncode == 0:
                            folder = result.stdout.strip()
                        break
                else:
                    import tkinter as tk
                    from tkinter import filedialog
                    root = tk.Tk()
                    root.withdraw()
                    folder = filedialog.askdirectory(title="Choose a folder to save results into")
                    root.destroy()
        except Exception as e:
            print(f"⚠️ Couldn't open a folder picker ({e}) -- type the path into the box above instead.")
            folder = None

        if folder:
            target_box.value = folder
    return _browse

# Glossary Accordion View
key_details = widgets.HTML(value='''
<div style="background-color: #f7f9fa; padding: 12px; border-left: 4px solid #2b5b84; border-radius: 4px; font-family: sans-serif; font-size: 13px; line-height: 1.5;">
    <b>📋 REGENERATION CONDITION DICTIONARY:</b><br>
    <b>Normal :</b> Uninjured control &nbsp;|&nbsp; <b>Anterior_regen :</b> Anterior regeneration &nbsp;|&nbsp;
    <b>Posterior_regen :</b> Posterior regeneration<br><br>
    Each species (<i>Syllis gracilis</i> / <i>Sphaerosyllis hystrix</i>) has its own independent Trinity
    assembly, gene IDs, and annotation report -- pick a species below before searching genes or running BLAST.
</div>
''')
glossary_dropdown = widgets.Accordion(children=[key_details], selected_index=None)
glossary_dropdown.set_title(0, "🔍 Click to view/hide Condition Glossary")
glossary_dropdown.layout.width = "670px"

# Inputs
species_dropdown = widgets.Dropdown(options=list(SPECIES_DATA.keys()), description="Species:", layout=widgets.Layout(width="420px"))
cmap_dropdown = widgets.Dropdown(options=list(CMAP_OPTIONS.keys()), description="Palette:", layout=widgets.Layout(width="420px"))

condition_multiselect = widgets.SelectMultiple(
    options=STANDARD_CONDITIONS,
    value=tuple(STANDARD_CONDITIONS),
    description="Conditions:",
    layout=widgets.Layout(width="200px", height="90px")
)

# Filter List: each species' sorted gene-ID list + lowercase mirror was precomputed once in the
# data-loading cell (SPECIES_DATA[...]["gene_ids_sorted"] / ["gene_ids_lower"]) instead of being
# rebuilt from scratch on every keystroke here -- Sgracilis alone has 309,576 genes, so re-sorting
# and re-lowering that on every character typed was the single biggest cause of this dashboard
# feeling unresponsive. The widget is also capped to MAX_FILTER_RESULTS entries so a broad search
# term never asks the browser to render hundreds of thousands of <option> rows at once.
MAX_FILTER_RESULTS = 500

search_filter = widgets.Text(placeholder="Type to filter genes (e.g. TR914...)...", description="Filter List:", layout=widgets.Layout(width="440px"))
gene_multiselect = widgets.SelectMultiple(options=SPECIES_DATA[species_dropdown.value]["gene_ids_sorted"][:MAX_FILTER_RESULTS], description="Select Genes:", layout=widgets.Layout(width="440px", height="140px"))
filter_hint = widgets.HTML(value="")

def _refresh_gene_options():
    species = SPECIES_DATA[species_dropdown.value]
    all_ids = species["gene_ids_sorted"]
    all_ids_lower = species["gene_ids_lower"]
    search_term = search_filter.value.strip().lower()
    if search_term:
        matches = [gid for gid, gid_lower in zip(all_ids, all_ids_lower) if search_term in gid_lower]
    else:
        matches = all_ids
    gene_multiselect.options = matches[:MAX_FILTER_RESULTS]
    if len(matches) > MAX_FILTER_RESULTS:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>Showing first {MAX_FILTER_RESULTS} of {len(matches)} matches -- refine your search to narrow further.</i>"
    else:
        filter_hint.value = f"<i style='font-size:11px;color:#666'>{len(matches)} matching entr{'y' if len(matches) == 1 else 'ies'}.</i>"

def on_species_change(change):
    _refresh_gene_options()
species_dropdown.observe(on_species_change, names="value")

def update_multiselect_options(change):
    _refresh_gene_options()
search_filter.observe(update_multiselect_options, names="value")

multi_gene_input = widgets.Textarea(placeholder="OR paste multiple Gene IDs here (e.g. TR91477|c3_g1, TR144725|c1_g1)...", description="Paste Box:", layout=widgets.Layout(width="440px", height="100px"))

# --- Option C: Search by Annotation Keyword -- finds every gene (in the currently selected
# species) whose Trinotate BLASTX description contains a keyword, then runs it through the exact
# same Heatmap -> Plots -> Annotation pipeline as the other two search options, sharing the same
# output_area below. ---
keyword_search_box = widgets.Text(placeholder="e.g. kinase, laminin, cytochrome...", description="Keyword:", layout=widgets.Layout(width="250px"))
keyword_search_button = widgets.Button(description="🔎 Search Annotations", button_style="warning", layout=widgets.Layout(width="250px", height="30px"))

MAX_KEYWORD_MATCHES = 30

# TWO SEPARATE BUTTON CONTROLS
submit_button = widgets.Button(description="🎯 Generate Visuals (View Only)", button_style="success", layout=widgets.Layout(width="330px", height="40px"))
export_button = widgets.Button(description="💾 Export Active Results (Save CSV/PDF)", button_style="info", layout=widgets.Layout(width="330px", height="40px"))
save_location_box = widgets.Text(placeholder="exported_results (default if left blank)", description="Save to:", layout=widgets.Layout(width="360px"))
browse_button = widgets.Button(description="📁 Browse...", layout=widgets.Layout(width="120px"))
browse_button.on_click(_make_browse_handler(save_location_box))

output_area = widgets.Output()

def on_submit_clicked(b):
    with output_area:
        clear_output(wait=True)
        chosen_genes = list(gene_multiselect.value)
        pasted_genes = [g.strip() for g in multi_gene_input.value.replace(",", " ").split() if g.strip()]
        final_gene_list = sorted(list(set(chosen_genes + pasted_genes)))

        if not final_gene_list:
            print("❌ Input target genes first!")
            return

        species = species_dropdown.value
        cmap = CMAP_OPTIONS[cmap_dropdown.value]

        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(species, final_gene_list, conditions_chosen=list(condition_multiselect.value))
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_CONDITIONS, SPECIES_DATA[species]["df"], "TMM-Normalized Expression", cmap)
        _render_bar_charts(cmap)
        _print_annotations(final_gene_list)

def on_export_clicked(b):
    with output_area:
        species = _CURRENT_SPECIES or species_dropdown.value
        cmap = CMAP_OPTIONS[cmap_dropdown.value]
        _base_dir = save_location_box.value.strip() or "exported_results"
        save_dir = os.path.join(_base_dir, datetime.now().strftime("search_%Y-%m-%d_%H-%M-%S"))
        print(f"📁 Saving this search to its own new folder: {save_dir}/")
        export_cached_results(save_dir=save_dir, cmap=cmap)
        save_multi_gene_views(_CURRENT_GENES, _CURRENT_CONDITIONS, SPECIES_DATA[species]["df"], "TMM-Normalized Expression", cmap, save_dir=save_dir)

def on_keyword_search_clicked(b):
    with output_area:
        clear_output(wait=True)
        keyword = keyword_search_box.value.strip()
        if not keyword:
            print("❌ Type a keyword first!")
            return

        species = species_dropdown.value
        matched_ids = _search_annotations_by_keyword(keyword, species)
        if not matched_ids:
            print(f"🛑 No genes found with '{keyword}' in their annotation for {species}.")
            return

        print(f"✅ Found {len(matched_ids)} gene(s) matching '{keyword}' in {species}.")
        if len(matched_ids) > MAX_KEYWORD_MATCHES:
            print(f"⚠️ Limiting to the first {MAX_KEYWORD_MATCHES} matches to keep the plots manageable.")
            matched_ids = matched_ids[:MAX_KEYWORD_MATCHES]

        cmap = CMAP_OPTIONS[cmap_dropdown.value]

        # Output order: Heatmap -> Plots -> Annotation
        plot_multi_genes(species, matched_ids, conditions_chosen=list(condition_multiselect.value))
        render_multi_gene_views(_CURRENT_GENES, _CURRENT_CONDITIONS, SPECIES_DATA[species]["df"], "TMM-Normalized Expression", cmap)
        _render_bar_charts(cmap)
        _print_annotations(matched_ids)

submit_button.on_click(on_submit_clicked)
export_button.on_click(on_export_clicked)
keyword_search_button.on_click(on_keyword_search_clicked)

# Layout Setup
print("📊 Ribeiro Regeneration Dual-Control Workspace Ready:")
display(widgets.VBox([
    glossary_dropdown,
    widgets.HTML("<br><b>Step 1: Choose a Species and Regeneration Conditions</b>"),
    widgets.HBox([widgets.VBox([species_dropdown, cmap_dropdown]), widgets.VBox([condition_multiselect])]),
    widgets.HTML("<br><b>Step 2: Choose Target Genes</b>"),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<i>Option A: Filter List</i>"), search_filter, gene_multiselect, filter_hint]),
        widgets.VBox([widgets.HTML("<i>Option B: Paste Box</i>"), multi_gene_input]),
        widgets.VBox([widgets.HTML("<i>Option C: Search by Keyword</i>"), keyword_search_box, keyword_search_button]),
    ]),
    widgets.HTML("<br><b>Step 3: Run and/or Save</b>"),
    widgets.HBox([save_location_box, browse_button]),
    widgets.HBox([submit_button, export_button]),
    output_area
]))